# Bid shading for first-price auctions (iPinYou)

Most programmatic ad buying moved from second-price to **first-price** auctions
around 2019: the winner now pays its own bid. Under second price, bidding an
impression's value is optimal, because what you pay is set by the competition.
Under first price, bidding your value wins at zero profit: every win overpays
by the gap between your bid and the price that would have won. So a
first-price bidder **shades** its bid below its value. How far depends on the
competition for that impression, which it has to learn.

For a request worth `v` with win probability `F(b)` at bid `b`, the
expected-surplus-maximising bid is

    b* = argmax_b (v - b) · F(b)

so the problem is learning `F(b | request)`, the **bid landscape**.

**What a first-price bidder actually observes:** only whether it won at the
price it bid. Never the price it needed to win: a loss says the winning price
was above its bid, a win that it was below. Both are censored. The landscape
models here are trained on exactly that and nothing more: each fit-day request
gets one exploration bid, drawn independently of the market price, and one
won/lost label. The market price itself is used only by the replay to decide
the outcome, as the exchange would.

**Simulated first price:** iPinYou ran second-price auctions, and each logged
impression carries its market price, the highest competing bid. A first-price
auction against the same competitors is won by a bid above that price, and
the winner pays its bid. This holds the other bidders' behaviour fixed, a
standard assumption in offline bid-shading studies; in a real switch they
would shade too.

**Strategies,** each with its own bid level tuned on 10 June and evaluated on
11-12 June, same budgets, same protocol as `ipinyou-replay.ipynb`:

| Strategy | Bid |
|---|---|
| `fp_flat` | one bid for every request |
| `fp_ev` | EV bid `level · pCTR / mean pCTR`, unshaded. Its tuned level is already the best *linear* shading: a constant fraction of value |
| `fp_shade_global` | `b*` under a landscape per advertiser, which ignores the request |
| `fp_shade_ctx` | `b*` under a landscape conditioned on the request's features |

`fp_ev` is the baseline to beat: shading only earns its keep if it does better
than simply tuning the EV level down. `sp_flat` and `sp_ev` replay the second
price rules as before and must reproduce `ipinyou-replay.ipynb` v2 exactly.

**Caveat** (shared with the CTR replay): logs exist only for auctions iPinYou
won, so its market prices are truncated at iPinYou's own bid.

Add the dataset `lastsummer/ipinyou` as an input before running. Runtime is
roughly 30-40 minutes on a Kaggle CPU, most of it tuning the shaded bid levels.

In [ ]:
import os, glob, json, time, hashlib
import numpy as np, pandas as pd
from pandas.api.types import union_categoricals
from scipy import sparse
from scipy.special import expit
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, log_loss

# Find the dataset wherever Kaggle mounted it (IPINYOU_ROOT overrides).
ROOT = os.environ.get("IPINYOU_ROOT") or next(
    r for r, dirs, _ in os.walk("/kaggle/input") if "training2nd" in dirs)
SEASON = os.path.join(ROOT, "training2nd")
OUT = os.environ.get("IPINYOU_OUT", "/kaggle/working")
print("dataset root:", ROOT)

COLS = ["bidid", "timestamp", "logtype", "ipinyouid", "useragent", "IP", "region",
        "city", "adexchange", "domain", "url", "urlid", "slotid", "slotwidth",
        "slotheight", "slotvisibility", "slotformat", "slotprice", "creative",
        "bidprice", "payprice", "keypage", "advertiser", "usertag"]
FIT_DAYS, TUNE_DAYS, TEST_DAYS = ["06", "07", "08", "09"], ["10"], ["11", "12"]

In [ ]:
# ── Load: identical to ipinyou-replay.ipynb, so its results reproduce ───────
def device_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    tablet = ua.str.contains("ipad|tablet") | (ua.str.contains("android") & ~ua.str.contains("mobile"))
    mobile = ua.str.contains("iphone|ipod|mobile|windows phone|symbian|blackberry")
    return pd.Series(np.select([tablet, mobile], ["tablet", "mobile"], "desktop"), index=ua.index)

def os_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    return pd.Series(np.select(
        [ua.str.contains("windows"), ua.str.contains("android"),
         ua.str.contains("iphone|ipad|ipod"), ua.str.contains("mac os"),
         ua.str.contains("linux")],
        ["windows", "android", "ios", "mac", "linux"], "other"), index=ua.index)

def browser_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    return pd.Series(np.select(
        [ua.str.contains("msie|trident"), ua.str.contains("chrome"),
         ua.str.contains("firefox"), ua.str.contains("safari"), ua.str.contains("opera")],
        ["ie", "chrome", "firefox", "safari", "opera"], "other"), index=ua.index)

clicked = set()
for f in sorted(glob.glob(os.path.join(SEASON, "clk.*.txt.bz2"))):
    clicked.update(pd.read_csv(f, sep="\t", header=None, usecols=[0], quoting=3,
                               dtype=str)[0])
print(f"{len(clicked):,} clicked bid ids")

CATS = ["advertiser", "adexchange", "region", "city", "domain", "slotsize",
        "slotvisibility", "slotformat", "device", "os", "browser", "hour", "day"]
TAG_INDEX, tag_rows, tag_cols, offset = {}, [], [], 0
frames = []
for f in sorted(glob.glob(os.path.join(SEASON, "imp.*.txt.bz2"))):
    t0 = time.time()
    d = pd.read_csv(f, sep="\t", header=None, names=COLS, quoting=3, dtype=str,
                    usecols=["bidid", "timestamp", "useragent", "region", "city",
                             "adexchange", "domain", "slotwidth", "slotheight",
                             "slotvisibility", "slotformat", "slotprice",
                             "payprice", "advertiser", "usertag"])
    out = pd.DataFrame({
        "ts": d["timestamp"].str[:14],
        "click": d["bidid"].isin(clicked).astype("int8"),
        "floor": pd.to_numeric(d["slotprice"], errors="coerce").fillna(0).astype("int32"),
        "pay": pd.to_numeric(d["payprice"], errors="coerce").fillna(0).astype("int32"),
    })
    out["hour"] = d["timestamp"].str[8:10]
    out["day"] = d["timestamp"].str[6:8]
    out["slotsize"] = d["slotwidth"] + "x" + d["slotheight"]
    out["device"], out["os"], out["browser"] = (
        device_of(d["useragent"]), os_of(d["useragent"]), browser_of(d["useragent"]))
    for c in ["advertiser", "adexchange", "region", "city", "domain",
              "slotvisibility", "slotformat"]:
        out[c] = d[c].fillna("na")
    for c in CATS:
        out[c] = out[c].astype("category")
    r, k = [], []
    for i, t in enumerate(d["usertag"].fillna("").to_numpy()):
        if t and t != "null":
            for tag in t.split(","):
                tag = tag.strip()
                if tag:
                    r.append(i)
                    k.append(TAG_INDEX.setdefault(tag, len(TAG_INDEX)))
    tag_rows.append(np.asarray(r, dtype=np.int64) + offset)
    tag_cols.append(np.asarray(k, dtype=np.int64))
    offset += len(d)
    frames.append(out)
    print(f"{os.path.basename(f)}: {len(out):,} rows, {out.click.sum()} clicks, {time.time() - t0:.0f}s")

df = pd.DataFrame({
    c: (union_categoricals([f[c] for f in frames]) if c in CATS
        else np.concatenate([f[c].to_numpy() for f in frames]))
    for c in frames[0].columns})
del frames
rows_, cols_ = np.concatenate(tag_rows), np.concatenate(tag_cols)
TAGS = sparse.csr_matrix((np.ones(len(rows_), dtype=np.float32), (rows_, cols_)),
                         shape=(offset, len(TAG_INDEX)))
TAGS.data[:] = 1.0
del rows_, cols_, tag_rows, tag_cols
df["row"] = np.arange(len(df))
df = df.sort_values("ts", kind="stable").reset_index(drop=True)
TAGS = TAGS[df["row"].to_numpy()]
df = df.drop(columns="row")
print(f"{len(df):,} impressions, {df.click.sum():,} clicks, {len(TAG_INDEX)} user tags")

In [ ]:
# ── The value of each request: v2's chosen CTR model, calibrated ─────────────
# Logistic regression with user tags, isotonic calibration on the tune day:
# the model the engine serves. Refit here with the same data and solver.
FEATURES = ["advertiser", "adexchange", "region", "city", "domain", "slotsize",
            "slotvisibility", "slotformat", "device", "os", "browser", "hour"]
fit_m = df.day.isin(FIT_DAYS).to_numpy()
tune_m = df.day.isin(TUNE_DAYS).to_numpy()
test_m = df.day.isin(TEST_DAYS).to_numpy()
fit, tune, test = (df[m].reset_index(drop=True) for m in (fit_m, tune_m, test_m))
TAGS_FIT, TAGS_TUNE, TAGS_TEST = TAGS[fit_m], TAGS[tune_m], TAGS[test_m]
print(f"fit {len(fit):,}  tune {len(tune):,}  test {len(test):,}")

ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, dtype=np.float32)
ohe.fit(fit[FEATURES].astype(str))

def lr_design(frame, tags):
    return sparse.hstack([ohe.transform(frame[FEATURES].astype(str)), tags], format="csr")

t0 = time.time()
ctr = LogisticRegression(C=0.1, max_iter=500).fit(lr_design(fit, TAGS_FIT), fit.click)
raw_tune = ctr.predict_proba(lr_design(tune, TAGS_TUNE))[:, 1]
raw_test = ctr.predict_proba(lr_design(test, TAGS_TEST))[:, 1]
iso = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1.0).fit(raw_tune, tune.click)
tune["p_cal"], test["p_cal"] = iso.predict(raw_tune), iso.predict(raw_test)
ctr_auc = roc_auc_score(test.click, raw_test)
print(f"CTR model {time.time() - t0:.0f}s, test AUC {ctr_auc:.5f}",
      "(v2: 0.80301)", "OK" if abs(ctr_auc - 0.8030108) < 1e-5 else "MISMATCH")

## The bid landscape, learned from won/lost only

Each request gets one exploration bid, log-uniform between 1 and 400 fen CPM
(the market prices run up to about 300) and drawn independently of the market
price, plus the outcome a first-price bidder would see: won if the bid beat
the market price. A real bidder buys this data by spending a small share of
its traffic on exploration bids.

Both landscapes are logistic in log bid, `F(b | x) = σ(a(x) + β(x) · ln b)`:

- **global:** `a` and `β` per advertiser only. The same curve for every request.
- **contextual:** `a(x)` from the request's features and user tags, `β(x)` per
  advertiser and ad exchange.

With `β > 0` the curve rises with the bid, and `ln((v - b) · F(b))` is concave
in `b`, so `b*` is unique and a bisection finds it exactly.

In [ ]:
# ── Landscape models ─────────────────────────────────────────────────────────
BID_LO, BID_HI = 1.0, 400.0
rng = np.random.default_rng(0)

def exploration(frame):
    bids = np.exp(rng.uniform(np.log(BID_LO), np.log(BID_HI), len(frame)))
    return bids, (bids > frame["pay"].to_numpy()).astype(np.int8)

fit_bid, fit_win = exploration(fit)
tune_bid, tune_win = exploration(tune)
print(f"exploration win rate: fit {fit_win.mean():.1%}, tune {tune_win.mean():.1%}")

SLOPE_BY = ["advertiser", "adexchange"]
w_ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100,
                      dtype=np.float32).fit(fit[FEATURES].astype(str))
s_ohe = OneHotEncoder(handle_unknown="ignore", dtype=np.float32).fit(fit[SLOPE_BY].astype(str))
a_ohe = OneHotEncoder(handle_unknown="ignore", dtype=np.float32).fit(fit[["advertiser"]].astype(str))

def landscape_design(frame, tags, bids, context):
    # float64 throughout: some scikit-learn versions fit float32 input in
    # float32, and the engine must reproduce a(x) to well within 1e-6.
    lb = np.log(bids).reshape(-1, 1)
    if context:
        X = sparse.hstack([w_ohe.transform(frame[FEATURES].astype(str)), tags,
                           sparse.csr_matrix(lb),
                           s_ohe.transform(frame[SLOPE_BY].astype(str)).multiply(lb)],
                          format="csr")
    else:
        A = a_ohe.transform(frame[["advertiser"]].astype(str))
        X = sparse.hstack([A, A.multiply(lb)], format="csr")
    return X.astype(np.float64)

LANDSCAPES = {}
for name, context in [("global", False), ("ctx", True)]:
    t0 = time.time()
    LANDSCAPES[name] = LogisticRegression(C=1.0, max_iter=500).fit(
        landscape_design(fit, TAGS_FIT, fit_bid, context), fit_win)
    print(f"{name} landscape fitted in {time.time() - t0:.0f}s")

def curve(name, frame, tags):
    """a(x) and β(x) per row: the decision function at ln b = 0 and ln b = 1."""
    m, context = LANDSCAPES[name], name == "ctx"
    a = m.decision_function(landscape_design(frame, tags, np.ones(len(frame)), context))
    beta = m.decision_function(landscape_design(frame, tags, np.full(len(frame), np.e), context)) - a
    return a, beta

for frame, tags, label in [(tune, TAGS_TUNE, "tune"), (test, TAGS_TEST, "test")]:
    for name in LANDSCAPES:
        a, beta = curve(name, frame, tags)
        print(f"{label} {name}: slope β min {beta.min():.3f}, median {np.median(beta):.3f}")
        # β <= 0 would make winning cheaper by bidding less. It is checked,
        # not assumed; a floor keeps the shading formula defined if it occurs.
        frame[f"a_{name}"], frame[f"b_{name}"] = a, np.maximum(beta, 0.05)

def win_ece(y, p):
    return sum(len(c) / len(p) * abs(p[c].mean() - y[c].mean())
               for c in np.array_split(np.argsort(p, kind="stable"), 10))

landscape_rows = []
for name in LANDSCAPES:
    p = expit(tune[f"a_{name}"].to_numpy() + tune[f"b_{name}"].to_numpy() * np.log(tune_bid))
    landscape_rows.append(dict(landscape=name, tune_auc=roc_auc_score(tune_win, p),
                               tune_logloss=log_loss(tune_win, p), tune_ece=win_ece(tune_win, p)))
landscape_table = pd.DataFrame(landscape_rows)
print(landscape_table.to_string(index=False))

# How well does each landscape know the price an impression needs? Its median
# predicted market price (F = 1/2) against the realised one, on the tune day.
for name in LANDSCAPES:
    med = np.exp(-tune[f"a_{name}"] / tune[f"b_{name}"])
    print(f"{name}: median absolute error of the predicted median price "
          f"{np.median(np.abs(med - tune.pay)):.1f} fen CPM")

In [ ]:
# ── Shading ──────────────────────────────────────────────────────────────────
def shade(v, a, beta, iters=24):
    """argmax over b in (0, v] of (v - b) · σ(a + β ln b).

    The optimum is the root of h(b) = β (1 - F(b)) (v - b) - b, which is
    positive near 0 and equals -v at b = v, with one crossing in between.
    """
    lo, hi = np.zeros_like(v), v.astype(float).copy()
    for _ in range(iters):
        mid = (lo + hi) / 2
        F = expit(a + beta * np.log(np.maximum(mid, 1e-9)))
        right = beta * (1 - F) * (v - mid) > mid
        lo, hi = np.where(right, mid, lo), np.where(right, hi, mid)
    return (lo + hi) / 2

# A brute-force check of the bisection on a grid, for a few thousand rows.
chk = test.sample(min(2000, len(test)), random_state=0)
v = np.full(len(chk), 120.0)
grid = np.linspace(0.01, 1, 2000)[None, :] * v[:, None]
obj = (v[:, None] - grid) * expit(chk.a_ctx.to_numpy()[:, None]
                                  + chk.b_ctx.to_numpy()[:, None] * np.log(grid))
b_grid = grid[np.arange(len(chk)), obj.argmax(1)]
b_bis = shade(v, chk.a_ctx.to_numpy(), chk.b_ctx.to_numpy())
print(f"bisection vs grid: max gap {np.max(np.abs(b_grid - b_bis)):.3f} fen (grid step {v[0] / 2000:.3f})")

In [ ]:
# ── Replay ───────────────────────────────────────────────────────────────────
def replay(bids, pay, clicks, budget, first_price):
    """Requests in time order: win when the bid beats the market price, pay the
    market price (second price) or the bid (first price), stop once the budget
    is spent."""
    win = bids > pay
    cost = np.where(win, bids if first_price else pay, 0)
    won = win & (np.cumsum(cost) <= budget)
    return int(won.sum()), int(clicks[won].sum()), float(cost[won].sum()), won

def by_advertiser(frame):
    cols = ["pay", "click", "p_cal", "a_global", "b_global", "a_ctx", "b_ctx"]
    return {adv: {**{c: g[c].to_numpy() for c in cols},
                  "block": (g["day"].astype(str) + g["hour"].astype(str)).to_numpy()}
            for adv, g in frame.groupby("advertiser", observed=True)}

TUNE, TEST = by_advertiser(tune), by_advertiser(test)
NORM = {adv: a["p_cal"].mean() for adv, a in TUNE.items()}   # tune-day mean pCTR

SP = ["sp_flat", "sp_ev"]
FP = ["fp_flat", "fp_ev", "fp_shade_global", "fp_shade_ctx"]
STRATEGIES = SP + FP

def bids_for(strategy, a, adv, level):
    if strategy.endswith("flat"):
        return np.full(len(a["pay"]), float(level))
    value = level * a["p_cal"] / NORM[adv]
    if strategy.startswith("fp_shade"):
        land = strategy.rsplit("_", 1)[1]
        return shade(value, a[f"a_{land}"], a[f"b_{land}"])
    return value

GRID_SP = np.arange(2, 402, 2)                                   # v2's grid
# First-price arms share one wider grid: a shaded bid is a fraction of its
# value, so the value level that spends the budget is higher.
GRID_FP = np.unique(np.round(np.geomspace(2, 4000, 60)))
BUDGET_FRACTIONS = {"1/32": 1 / 32, "1/8": 1 / 8, "1/2": 1 / 2}

def tune_levels(data, grid_for):
    """Each strategy's best level per advertiser and budget: the most clicks,
    ties to the cheaper. Bids depend on the level, not the budget, so they are
    computed once per level and replayed at every budget."""
    levels = {}
    for adv, a in data.items():
        for s in STRATEGIES:
            first = s.startswith("fp")
            best = {}
            for b in grid_for(s):
                bids = bids_for(s, a, adv, b)
                for name, frac in BUDGET_FRACTIONS.items():
                    _, clk, cost, _ = replay(bids, a["pay"], a["click"],
                                             frac * a["pay"].sum(), first)
                    key = (clk, -cost)
                    if name not in best or key > best[name][0]:
                        best[name] = (key, b)
            for name in BUDGET_FRACTIONS:
                levels[adv, s, name] = best[name][1]
    return levels

grid_for = lambda s: GRID_FP if s.startswith("fp") else GRID_SP
t0 = time.time()
levels = tune_levels(TUNE, grid_for)
print(f"levels tuned in {time.time() - t0:.0f}s")

# The test-day bids at the tuned levels, kept for the bootstrap.
TEST_BIDS = {(adv, s, name): bids_for(s, a, adv, levels[adv, s, name])
             for adv, a in TEST.items() for s in STRATEGIES for name in BUDGET_FRACTIONS}

rows = []
for adv, a in TEST.items():
    for name, frac in BUDGET_FRACTIONS.items():
        for s in STRATEGIES:
            bids = TEST_BIDS[adv, s, name]
            imps, clk, cost, won = replay(bids, a["pay"], a["click"],
                                          frac * a["pay"].sum(), s.startswith("fp"))
            rows.append(dict(advertiser=adv, budget=name, strategy=s,
                             level=float(levels[adv, s, name]), imps=imps, clicks=clk, cost=cost,
                             # what wins paid above the price that would have won
                             overpaid=float((bids[won] - a["pay"][won]).sum())
                             if s.startswith("fp") else 0.0))
results = pd.DataFrame(rows)
results["ecpc"] = results.cost / results.clicks.replace(0, np.nan)
print(results.pivot_table(index=["advertiser", "budget"], columns="strategy",
                          values="clicks").to_string())

In [ ]:
# ── Uncertainty: block bootstrap over test hours, paired ─────────────────────
rng = np.random.default_rng(0)
B = 300
BLOCKS = {adv: [np.flatnonzero(a["block"] == k) for k in np.unique(a["block"])]
          for adv, a in TEST.items()}

def pooled(sample):
    out = {}
    for adv, idx in sample.items():
        pay, click = TEST[adv]["pay"][idx], TEST[adv]["click"][idx]
        for name, frac in BUDGET_FRACTIONS.items():
            for s in STRATEGIES:
                _, clk, cost, _ = replay(TEST_BIDS[adv, s, name][idx], pay, click,
                                         frac * pay.sum(), s.startswith("fp"))
                out[name, s, "clicks"] = out.get((name, s, "clicks"), 0) + clk
                out[name, s, "cost"] = out.get((name, s, "cost"), 0) + cost
    return out

observed = pooled({adv: np.concatenate(b) for adv, b in BLOCKS.items()})
boot = [pooled({adv: np.concatenate([b[i] for i in np.sort(rng.integers(0, len(b), len(b)))])
                for adv, b in BLOCKS.items()}) for _ in range(B)]

def lift(o, name, s, base):
    if not o[name, base, "clicks"]:
        return np.nan
    return o[name, s, "clicks"] / o[name, base, "clicks"] - 1

def ecpc_change(o, name, s, base):
    if not (o[name, s, "clicks"] and o[name, base, "clicks"]):
        return np.nan
    return ((o[name, s, "cost"] / o[name, s, "clicks"])
            / (o[name, base, "cost"] / o[name, base, "clicks"]) - 1)

def ci(values):
    return [float(np.nanquantile(values, 0.025)), float(np.nanquantile(values, 0.975))]

summary_rows = []
for name in BUDGET_FRACTIONS:
    pooled_res = results[results.budget == name].groupby("strategy")
    for s in FP:
        row = dict(budget=name, strategy=s, clicks=observed[name, s, "clicks"],
                   sp_ev_clicks=observed[name, "sp_ev", "clicks"],
                   overpaid_share=float(pooled_res.overpaid.sum()[s] / pooled_res.cost.sum()[s]))
        for base in ("fp_ev", "fp_flat"):
            if base == s:
                continue
            row[f"lift_vs_{base}"] = lift(observed, name, s, base)
            row[f"lift_vs_{base}_ci"] = ci([lift(o, name, s, base) for o in boot])
            row[f"ecpc_vs_{base}"] = ecpc_change(observed, name, s, base)
            row[f"ecpc_vs_{base}_ci"] = ci([ecpc_change(o, name, s, base) for o in boot])
        summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.float_format", "{:.3f}".format)
print(summary_df[["budget", "strategy", "clicks", "sp_ev_clicks", "overpaid_share",
                  "lift_vs_fp_ev", "lift_vs_fp_ev_ci", "lift_vs_fp_flat"]].to_string(index=False))

# ipinyou-replay.ipynb v2 published these second-price numbers (flat, ev).
V2 = {"1/32": (127, 856), "1/8": (406, 1606), "1/2": (1562, 2268)}
for name, expected in V2.items():
    got = (observed[name, "sp_flat", "clicks"], observed[name, "sp_ev", "clicks"])
    print(f"v2 check {name}: expected {expected}, got {got}", "OK" if got == expected else "MISMATCH")

In [ ]:
# ── Robustness: every strategy at its best level in hindsight ────────────────
t0 = time.time()
hindsight_levels = tune_levels(TEST, grid_for)
hindsight = {}
for adv, a in TEST.items():
    for name, frac in BUDGET_FRACTIONS.items():
        for s in STRATEGIES:
            _, clk, _, _ = replay(bids_for(s, a, adv, hindsight_levels[adv, s, name]),
                                  a["pay"], a["click"], frac * a["pay"].sum(), s.startswith("fp"))
            hindsight[name, s] = hindsight.get((name, s), 0) + clk
print(f"hindsight levels in {time.time() - t0:.0f}s")
for name in BUDGET_FRACTIONS:
    print(f"budget {name}: " + ", ".join(
        f"{s} {hindsight[name, s]}" + (f" ({hindsight[name, s] / hindsight[name, 'fp_ev'] - 1:+.1%} vs fp_ev)"
                                      if s.startswith("fp_shade") else "")
        for s in STRATEGIES))

In [ ]:
# ── Export ───────────────────────────────────────────────────────────────────
# 1. bid_shading_results.json: everything above.
# 2. ipinyou_win_model.json: the contextual landscape for the engine, in the
#    same layout as ipinyou_ctr_model.json (feature weights keyed by value,
#    infrequent buckets, user-tag weights) plus the slope terms, the tuned
#    first-price levels, and parity examples the engine's tests must reproduce.
with open(f"{OUT}/bid_shading_results.json", "w") as f:
    json.dump({
        "version": 1,
        "data": "iPinYou season 2 (training2nd), 5 advertisers, 6-12 June 2013",
        "auction": "simulated first price: win when the bid beats the logged market price, pay the bid",
        "splits": {"fit": FIT_DAYS, "tune": TUNE_DAYS, "test": TEST_DAYS},
        "ctr_test_auc": ctr_auc,
        "exploration": {"distribution": "log-uniform", "low": BID_LO, "high": BID_HI,
                        "fit_win_rate": float(fit_win.mean())},
        "landscapes": landscape_table.to_dict("records"),
        "bootstrap": {"blocks": "test hours", "replicates": B},
        "summary": summary_df.to_dict("records"),
        "hindsight_clicks": [{"budget": n, "strategy": s, "clicks": hindsight[n, s]}
                             for n in BUDGET_FRACTIONS for s in STRATEGIES],
        "per_advertiser": results.to_dict("records"),
    }, f, indent=2, default=float)

m = LANDSCAPES["ctx"]
coef = m.coef_[0]
names = w_ohe.get_feature_names_out()
n_w, n_t = len(names), len(TAG_INDEX)
weights, infrequent = {f: {} for f in FEATURES}, {}
for name, w in zip(names, coef[:n_w]):
    feature = max((f for f in FEATURES if name.startswith(f + "_")), key=len)
    value = name[len(feature) + 1:]
    if value == "infrequent_sklearn":
        infrequent[feature] = float(w)
    else:
        weights[feature][value] = float(w)
tag_names = sorted(TAG_INDEX, key=TAG_INDEX.get)
tag_weights = {t: float(w) for t, w in zip(tag_names, coef[n_w:n_w + n_t])}
slope = {"base": float(coef[n_w + n_t]), **{f: {} for f in SLOPE_BY}}
for name, w in zip(s_ohe.get_feature_names_out(), coef[n_w + n_t + 1:]):
    feature = max((f for f in SLOPE_BY if name.startswith(f + "_")), key=len)
    slope[feature][name[len(feature) + 1:]] = float(w)
assert n_w + n_t + 1 + sum(len(slope[f]) for f in SLOPE_BY) == len(coef)

scoring = {"intercept": float(m.intercept_[0]), "weights": weights, "infrequent": infrequent,
           "tag_weights": tag_weights, "slope": slope, "slope_floor": 0.05}
version = hashlib.sha256(json.dumps(scoring, sort_keys=True).encode()).hexdigest()[:12]

# Parity examples: real test requests with this notebook's curve and shaded bid.
idx = np.random.default_rng(2).choice(len(test), min(500, len(test)), replace=False)
indptr, indices = TAGS_TEST.indptr, TAGS_TEST.indices
advs = test.advertiser.astype(str).to_numpy()[idx]
value = (np.array([levels[adv, "fp_shade_ctx", "1/2"] for adv in advs])
         * test.p_cal.to_numpy()[idx] / np.array([NORM[adv] for adv in advs]))
parity = pd.DataFrame({f: test[f].astype(str).to_numpy()[idx] for f in FEATURES})
parity["usertags"] = [",".join(tag_names[j] for j in indices[indptr[i]:indptr[i + 1]]) for i in idx]
parity["a"] = test.a_ctx.to_numpy()[idx]
parity["beta"] = test.b_ctx.to_numpy()[idx]
parity["value"] = value
parity["shaded_bid"] = shade(value, parity.a.to_numpy(), parity.beta.to_numpy())

win_model = {
    "name": "ipinyou-landscape-ctx",
    "version": version,
    "trained_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "source": "iPinYou RTB dataset, season 2; trained on won/lost outcomes of "
              "log-uniform exploration bids only",
    "form": "P(win | x, b) = sigmoid(a(x) + beta(x) * ln b); a(x) = intercept + weights + tags; "
            "beta(x) = max(slope.base + slope.advertiser + slope.adexchange, slope_floor)",
    "features": FEATURES, "slope_by": SLOPE_BY,
    **scoring,
    "price_unit": "CNY fen per thousand impressions (CPM)",
    "bidding": {
        "budget": "1/2",
        "rule": "value = level * pCTR / mean_pctr; bid = argmax_b (value - b) * P(win | x, b)",
        "advertisers": {adv: {"shade_level": float(levels[adv, "fp_shade_ctx", "1/2"]),
                              "fp_ev_level": float(levels[adv, "fp_ev", "1/2"]),
                              "mean_pctr": float(NORM[adv])} for adv in TEST},
    },
    "metrics": {"landscapes": landscape_table.to_dict("records"),
                "summary": summary_df.to_dict("records")},
    "parity": parity.to_dict("records"),
}
with open(f"{OUT}/ipinyou_win_model.json", "w") as f:
    json.dump(win_model, f, default=float)
print(f"wrote bid_shading_results.json and ipinyou_win_model.json ({version}): "
      f"{sum(map(len, weights.values()))} weights, {len(tag_weights)} tags, "
      f"{sum(len(slope[f]) for f in SLOPE_BY)} slope terms, {len(parity)} parity rows")